# Chip packing lines: variance test from a shift report

**Test for Homogeneity of Variance** · a worked example from the StatNex teaching guide

**The story.** A snack factory fills 150 g bags of chips on two packing lines, and 14 bags were weighed from each line. The shift report gives only the number of bags and the standard deviation of bag weight, rounded: 0.65 g for Line A and 1.965 g for Line B.

**The question.** Can the shift report alone show that Line B is less consistent?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `03_two_sample/homogeneity_of_variance__summary.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Variable,Line A,Line B
Sample size,14.0,14.0
Standard deviation,0.65,1.965

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
groups = ["Line A", "Line B"]
confidence = 95 / 100
alpha = 5 / 100

## Descriptive statistics

In [ ]:
# Summary data: each column holds the sample size and the standard deviation, top
# to bottom. An F ratio never needs the means, so the summary does not carry them.
rows = []
for name in groups:
    size, sd = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    rows.append([int(round(size)), np.nan, sd**2])
descriptive = pd.DataFrame(rows, index=groups, columns=["n", "mean", "variance"])
descriptive["SD"] = np.sqrt(descriptive["variance"])
descriptive["SE"] = descriptive["SD"] / np.sqrt(descriptive["n"])
print(descriptive)

## Ratio of variances and its confidence interval

In [ ]:
# The larger variance goes on top, so the ratio is at least 1. If the two groups were
# equally spread, it would land near 1; the interval is the range of true ratios the
# data are consistent with. SciPy has no one-line F test for two variances, so the F
# distribution is used directly.
(n1, n2), (v1, v2) = descriptive["n"], descriptive["variance"]
if v2 > v1:
    ratio, df1, df2 = v2 / v1, n2 - 1, n1 - 1
else:
    ratio, df1, df2 = v1 / v2, n1 - 1, n2 - 1
quantile = 0.5 + confidence / 2
lower = ratio / stats.f.ppf(quantile, df1, df2)
upper = ratio * stats.f.ppf(quantile, df2, df1)
print(f"\nRatio of variances = {ratio:.6g}")
print(f"{confidence:.0%} confidence interval: {lower:.6g} to {upper:.6g}")

## F test for equal variances (two-sided)

In [ ]:
p = min(1.0, 2 * min(stats.f.sf(ratio, df1, df2), stats.f.cdf(ratio, df1, df2)))
# The F test assumes normal data in both groups, and is very sensitive to heavy
# tails: with them it reports "different spreads" far too often. stats.levene(x, y,
# center="mean") (Levene) or center="median" (Brown-Forsythe) is the robust check.
# Do not use this test to choose between the pooled and Welch t tests: use Welch's.
# Power: the chance the two-sided test at alpha rejects if the true ratio equals the
# one observed. Computed from the observed ratio, it adds nothing to the P value; plan
# a study's power from a ratio that matters, chosen in advance.
upper_critical = stats.f.ppf(1 - alpha / 2, df1, df2)
lower_critical = stats.f.ppf(alpha / 2, df1, df2)
power = stats.f.sf(upper_critical / ratio, df1, df2) + stats.f.cdf(lower_critical / ratio, df1, df2)
test = pd.DataFrame([[ratio, df1, df2, p, power]], columns=["F", "df1", "df2", "P", "power"])
print()
print(test.to_string(index=False))

## What the result says

The shift report gives only 14 bags per line and the rounded standard deviations: 0.65 g for Line A and 1.965 g for Line B. The standard deviation is the typical distance of a bag from the average.

That is enough for the test. The ratio of the variances (the standard deviations squared) is the F statistic, 9.14. The P value is 0.0003, the chance of a ratio at least this far from 1 if both lines had the same spread. The 95% confidence interval for the true ratio runs from 2.93 to 28.5. Rounding moved the numbers only a little from the raw-weight result.

So the report alone shows that Line B is far less consistent. But it gives no means, so it says nothing about the average weights, or about the shape of the data the test relies on.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.